# X comet and BLEU

In [ ]:
!pip install unbabel-comet sacrebleu

In [1]:
from google.colab import files
import csv
from comet import download_model, load_from_checkpoint
from sacrebleu import corpus_bleu, sentence_bleu

In [2]:
class TSVLoader:
    def __init__(self, source_path, target_path, hyp_source_path, hyp_target_path, has_header=False):
        self.source_path = source_path
        self.target_path = target_path
        self.hyp_source_path = hyp_source_path
        self.hyp_target_path = hyp_target_path
        self.has_header = has_header

        # Lists that will store loaded lines
        self.source = []
        self.target = []
        self.hyp_source = []
        self.hyp_target = []

    def load(self):
        # Load source.tsv
        with open(self.source_path, "r", encoding="utf-8") as f:
            lines = f.read().splitlines()
            if self.has_header:
                lines = lines[1:]
            self.source = lines

        # Load target.tsv
        with open(self.target_path, "r", encoding="utf-8") as f:
            lines = f.read().splitlines()
            if self.has_header:
                lines = lines[1:]
            self.target = lines

        # Load hyp_source.tsv
        with open(self.hyp_source_path, "r", encoding="utf-8") as f:
            lines = f.read().splitlines()
            if self.has_header:
                lines = lines[1:]
            self.hyp_source = lines

        # Load hyp_target.tsv
        with open(self.hyp_target_path, "r", encoding="utf-8") as f:
            lines = f.read().splitlines()
            if self.has_header:
                lines = lines[1:]
            self.hyp_target = lines

    def print_statistics(self):
        print(f"Loaded {len(self.source)} source lines.")
        print(f"Loaded {len(self.target)} target lines.")
        print(f"Loaded {len(self.hyp_source)} hyp_source lines.")
        print(f"Loaded {len(self.hyp_target)} hyp_target lines.")

In [3]:
uploaded = files.upload()

Saving target.tsv to target.tsv
Saving source.tsv to source.tsv
Saving hyp_target.tsv to hyp_target.tsv
Saving hyp_source.tsv to hyp_source.tsv


In [4]:
loader = TSVLoader(
    "source.tsv",
    "target.tsv",
    "hyp_source.tsv",
    "hyp_target.tsv",
    has_header=False
)

loader.load()
loader.print_statistics()

Loaded 99 source lines.
Loaded 99 target lines.
Loaded 99 hyp_source lines.
Loaded 99 hyp_target lines.


In [5]:
class XCometScorer:
    def __init__(self, model_name="Unbabel/wmt22-comet-da"):
        print("Loading COMET model...")
        model_path = download_model(model_name)
        self.model = load_from_checkpoint(model_path)

    def score(self, src, mt, ref):
        """
        src = source sentences
        mt  = system hypotheses
        ref = gold references
        """
        data = [
            {"src": s, "mt": h, "ref": r}
            for s, h, r in zip(src, mt, ref)
        ]
        scores = self.model.predict(data, batch_size=8, gpus=0)
        return scores["scores"]

In [6]:
class BLEUScorer:
    def score(self, hyp, ref):
        scores = []
        for h, r in zip(hyp, ref):
            bleu = sentence_bleu(h, [r]).score
            scores.append(bleu)
        return scores


In [7]:
comet = XCometScorer()
bleu = BLEUScorer()

Loading COMET model...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

.gitattributes: 0.00B [00:00, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.10M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/616 [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/pytorch_lightning/core/saving.py:195: Found keys that are not in the model state dict but in the checkpoint: ['encoder.model.embeddings.position_ids']


In [8]:
# COMET evaluation
comet_scores = comet.score(
    src=loader.source,
    mt=loader.hyp_target,   # SYSTEM OUTPUT
    ref=loader.target       # GOLD REFERENCE
)

# BLEU evaluation
bleu_scores = bleu.score(
    hyp=loader.hyp_target,
    ref=loader.target
)

INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
INFO:pytorch_lightning.utilities.rank_zero:GPU available: False, used: False
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
Predicting DataLoader 0: 100%|██████████| 13/13 [02:07<00:00,  9.80s/it]


# Save Results

In [9]:
def save_scores_only_tsv(save_path, bleu_scores, comet_scores):
    with open(save_path, "w", encoding="utf-8", newline="") as f:
        writer = csv.writer(f, delimiter="\t")

        # Header
        writer.writerow(["id", "bleu", "comet"])

        # Rows
        for i, (bleu, comet) in enumerate(zip(bleu_scores, comet_scores)):
            writer.writerow([i, bleu, comet])

In [10]:
save_scores_only_tsv(
    "scores_bleu_and_comet.tsv",
    bleu_scores=bleu_scores,
    comet_scores=comet_scores
)

print("\nSCORES")
for i, (b, c) in enumerate(zip(bleu_scores, comet_scores)):
    print(f"ID {i} | BLEU: {b:.2f} | COMET: {c:.4f}")

print("\nSaved")


SCORES
ID 0 | BLEU: 79.56 | COMET: 0.9742
ID 1 | BLEU: 84.09 | COMET: 0.9678
ID 2 | BLEU: 24.88 | COMET: 0.7164
ID 3 | BLEU: 15.91 | COMET: 0.8388
ID 4 | BLEU: 8.12 | COMET: 0.8572
ID 5 | BLEU: 41.62 | COMET: 0.9260
ID 6 | BLEU: 100.00 | COMET: 0.9820
ID 7 | BLEU: 21.31 | COMET: 0.8013
ID 8 | BLEU: 13.35 | COMET: 0.9309
ID 9 | BLEU: 64.44 | COMET: 0.9163
ID 10 | BLEU: 100.00 | COMET: 0.9898
ID 11 | BLEU: 7.50 | COMET: 0.8465
ID 12 | BLEU: 43.74 | COMET: 0.8479
ID 13 | BLEU: 5.52 | COMET: 0.4909
ID 14 | BLEU: 0.00 | COMET: 0.4957
ID 15 | BLEU: 1.10 | COMET: 0.4212
ID 16 | BLEU: 0.00 | COMET: 0.2121
ID 17 | BLEU: 0.00 | COMET: 0.2198
ID 18 | BLEU: 10.68 | COMET: 0.5346
ID 19 | BLEU: 6.87 | COMET: 0.3457
ID 20 | BLEU: 3.75 | COMET: 0.5608
ID 21 | BLEU: 4.80 | COMET: 0.4565
ID 22 | BLEU: 6.77 | COMET: 0.4872
ID 23 | BLEU: 0.00 | COMET: 0.6442
ID 24 | BLEU: 3.09 | COMET: 0.5111
ID 25 | BLEU: 6.08 | COMET: 0.8619
ID 26 | BLEU: 0.00 | COMET: 0.4691
ID 27 | BLEU: 0.00 | COMET: 0.4530
ID 28 | 